In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score
from sklearn.model_selection import KFold, TimeSeriesSplit
from matplotlib import pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn 
from torch.utils.data import DataLoader

from src.config import SEED, TARGET, TIME_COL, DAGSHUB_USERNAME,DAGSHUB_REPO , processed_dataset, artifacts_path
from src.data.DLRM_preprocessing import DLRMPreprocessor
from src.models.dlrm_trainer import train_single_dlrm, set_seed
from src.models.dlrm import DLRM, FraudDLRMDataset

#from google.colab import drive
import os
import mlflow
import mlflow.pytorch
import dagshub
from dotenv import load_dotenv

c:\Users\ADMIN\Documents\DataScience\ieee-fraud-detection\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#%pip install -q mlflow dagshub

In [3]:
# Mount Google Drive into the remote Colab container
# drive.mount('/content/drive')

# SAVE_DIR = "/content/drive/MyDrive/IEEE-fraud-detection"
# os.makedirs(SAVE_DIR, exist_ok=True)

# 2. Load credentials directly from Google Drive .env file
# ENV_PATH = os.path.join(SAVE_DIR, ".env")
# load_dotenv(ENV_PATH)

dagshub.init(
    repo_owner=DAGSHUB_USERNAME,
    repo_name=DAGSHUB_REPO,
    mlflow=True
)

#mlflow.set_experiment("IEEE-fraud-multimodal-nn")
print(f"MLflow tracking set to DagsHub")

Accessing as anamish05

Initialized MLflow to track repo "anamish05/ieee-fraud-detection"

Repository anamish05/ieee-fraud-detection initialized!

MLflow tracking set to DagsHub


In [4]:

# Path inside mounted Google Drive
# DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/IEEE-fraud-detection")
# PROCESSED_PATH = DRIVE_PROJECT_ROOT / "data/processed/features_static_after_split_TE.parquet"

# Verify file exists before loading
# assert PROCESSED_PATH.exists(), f"File not found at: {PROCESSED_PATH}"

df = pd.read_parquet(processed_dataset)
print(f"Loaded engineered dataset: {df.shape}")

Loaded engineered dataset: (590540, 327)


In [5]:
# time-based folds training
n_splits = 5
tscv = TimeSeriesSplit(n_splits=n_splits)

# Arrays to store out-of-fold (OOF) predictions and evaluation scores
oof_predictions = np.zeros(len(df), dtype=np.float32)
oof_mask = np.zeros(len(df), dtype=bool)
fold_roc_scores = []
fold_pr_scores = []
fold_precision_scores=[]
fold_recall_scores=[]


Prepare data for NN - impute NaNs, scale features

In [6]:
te_features = ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M4']
te_cols = [c for c in df.columns if c.endswith('_target_enc')]
drop_cols = [
    TARGET, 
    TIME_COL, 
    'Pseudo_UID',
    'id_18', 'id_21', 'id_22', 'id_23', 'id_24', 'id_25', 'id_26' # sparse IP/proxy with >95% NaN
] + te_cols  # Drop target encoding columns, DLRM doesn't need them

features = [c for c in df.columns if c not in drop_cols]

cat_cols=['card1', 'card2', 'card3', 'card5',
                        'addr1', 'addr2',
                        'DeviceInfo',
                        'id_30', 'id_31', 'id_33', 'id_13', 'id_14', 'id_17', 'id_19', 'id_20', 'id_32', 'id_34'] + te_features

bin_cols=['M1', 'M2', 'M3', 'M5', 'M6', 'M7', 'M8', 'M9', 
          'id_12', 'id_15', 'id_16', 'id_27', 'id_28', 'id_29', 'id_35', 'id_36', 'id_37', 'id_38', 
          'DeviceType']

block_bin_flags = [c for c in ['block_2_present', 'block_3_present', 'block_4_present'] if c in df.columns]

freq_cols = [c for c in df.columns if c.endswith('_freq')] # apply log1p before scaling

special_excluded = set(drop_cols + cat_cols + bin_cols + block_bin_flags + freq_cols + ['uid_dt_diff'])
numeric_cols = [c for c in df.columns if c not in special_excluded]

In [7]:
batch_size = 512 if torch.device.type == "cpu" else 1024
eval_batch_size = 1024 if torch.device.type == "cpu" else 2048
device="cpu"
thresh=0.2

In [8]:
mlflow.set_experiment("IEEE-fraud-DLRM-time-folds")

2026/10/24 00:09:39 INFO mlflow.tracking.fluent: Experiment with name 'IEEE-fraud-DLRM-time-folds' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/1921b73747cf41238a1c40e86b3e98f4', creation_time=1792793379704, effective_trace_archival_retention=None, experiment_id='5', last_update_time=1792793379704, lifecycle_stage='active', name='IEEE-fraud-DLRM-time-folds', tags={}, trace_location=None, workspace='default'>

In [9]:
with mlflow.start_run(run_name="dlrm_5fold_timeseries"):
    for fold, (train_idx, val_idx) in enumerate(tscv.split(df)):
        print(f"\n{'='*30} Fold {fold + 1}/{n_splits} {'='*30}")
        print(f"Train size: {len(train_idx):,} | Val size: {len(val_idx):,}")

        X_train_fold = df.iloc[train_idx][features]
        X_val_fold   = df.iloc[val_idx][features]
        y_train_fold = df.iloc[train_idx][TARGET].values.astype(np.float32)
        y_val_fold   = df.iloc[val_idx][TARGET].values.astype(np.float32)

        #  Fit Preprocessor STRICTLY on this fold's train split
        preprocessor = DLRMPreprocessor(
            numeric_cols=numeric_cols,
            freq_cols=freq_cols,
            cat_cols=cat_cols,
            bin_cols=bin_cols,
            block_bin_flags=block_bin_flags,
            min_freq=50,
            scaler_type='robust',
            clip_val=5.0
        )
        
        X_tr_cont, X_tr_cat = preprocessor.fit_transform(X_train_fold)
        X_va_cont, X_va_cat = preprocessor.transform(X_val_fold)

        # Save this fold's preprocessor
        preprocessor.save(os.path.join(artifacts_path, f"preprocessor_fold{fold}.joblib"))

        # DataLoaders
        train_loader = DataLoader(
            FraudDLRMDataset(X_tr_cont, X_tr_cat, y_train_fold),
            batch_size=batch_size,
            shuffle=True,
            drop_last=True
        )
        val_loader = DataLoader(
            FraudDLRMDataset(X_va_cont, X_va_cat, y_val_fold),
            batch_size=eval_batch_size,
            shuffle=False
        )

        # Instantiate Model & Loss with fold-specific pos_weight
        n_neg, n_pos = (y_train_fold == 0).sum(), (y_train_fold == 1).sum()
        pos_weight = torch.tensor([float(n_neg / n_pos) ** 0.5], dtype=torch.float32, device=device)
        criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

        set_seed(SEED + fold)
        model = DLRM(
            n_dense=X_tr_cont.shape[1],
            cardinalities=preprocessor.cardinalities,
            d=16
        ).to(device)

        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='max', factor=0.5, patience=1, min_lr=1e-5
        )

        # Train with PR-AUC Early Stopping
        ckpt_path = os.path.join(artifacts_path, f"dlrm_fold_{fold}.pt")
        trained_model, _, val_preds = train_single_dlrm(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            criterion=criterion,
            optimizer=optimizer,
            scheduler=scheduler,
            epochs=15,
            patience=3,
            device=device,
            save_path=ckpt_path
        )

        #  Record Out-of-Fold (OOF) Metrics
        fold_pr = average_precision_score(y_val_fold, val_preds)
        fold_roc = roc_auc_score(y_val_fold, val_preds)
        fold_precision = precision_score(y_val_fold, (val_preds >= thresh).astype(int), zero_division=0)
        fold_recall = recall_score(y_val_fold, (val_preds >= thresh).astype(int), zero_division=0)

        fold_pr_scores.append(fold_pr)
        fold_roc_scores.append(fold_roc)
        fold_precision_scores.append(fold_precision)
        fold_recall_scores.append(fold_recall)

        oof_predictions[val_idx] = val_preds
        oof_mask[val_idx] = True

        print(f"Fold {fold + 1} Best | PR-AUC: {fold_pr:.5f} | ROC-AUC: {fold_roc:.5f} | Precision: {fold_precision:.5f} | Recall: {fold_recall:.5f}")

    #  Overall Out-Of-Fold Evaluation
    overall_y = df.iloc[oof_mask][TARGET].values
    overall_preds = oof_predictions[oof_mask]

    overall_pr_auc = average_precision_score(overall_y, overall_preds)
    overall_roc_auc = roc_auc_score(overall_y, overall_preds)
    overall_precision = precision_score(overall_y, (overall_preds >= thresh).astype(int), zero_division=0)
    overall_recall = recall_score(overall_y, (overall_preds >= thresh).astype(int), zero_division=0)

    mlflow.log_metrics({
        "mean_fold_pr_auc": np.mean(fold_pr_scores),
        "mean_fold_roc_auc": np.mean(fold_roc_scores),
        "mean_fold_precision": np.mean(fold_precision_scores),
        "mean_fold_recall": np.mean(fold_recall_scores),
        "overall_oof_pr_auc": overall_pr_auc,
        "overall_oof_roc_auc": overall_roc_auc,
        "overall_oof_precison": overall_precision,
        "overall_oof_recall": overall_recall
    })

    print(f"\n{'='*60}")
    print(f"CV Mean PR-AUC:  {np.mean(fold_pr_scores):.5f} +/- {np.std(fold_pr_scores):.5f}")
    print(f"CV Mean ROC-AUC: {np.mean(fold_roc_scores):.5f} +/- {np.std(fold_roc_scores):.5f}")
    print(f"CV Mean Precision: {np.mean(fold_precision_scores):.5f} +/- {np.std(fold_precision_scores):.5f}")
    print(f"CV Mean Recall: {np.mean(fold_recall_scores):.5f} +/- {np.std(fold_recall_scores):.5f}")
    print(f"Overall OOF PR-AUC:  {overall_pr_auc:.5f}")
    print(f"Overall OOF ROC-AUC: {overall_roc_auc:.5f}")
    print(f"Overall OOF Precision: {overall_precision:.5f}")
    print(f"Overall OOF Recall: {overall_recall:.5f}")
    print(f"{'='*60}")


============================== Fold 1/5 ==============================
Train size: 98,425 | Val size: 98,423
Epoch 01/15 | Train Loss: 0.3517 | Val Loss: 0.3668 | Val PR-AUC: 0.46879 | Val ROC-AUC: 0.84823Val Precision: 0.22229 | Val Recall: 0.58936
  --> Best checkpoint saved (PR-AUC: 0.46879)
Epoch 02/15 | Train Loss: 0.2460 | Val Loss: 0.3754 | Val PR-AUC: 0.48532 | Val ROC-AUC: 0.86149Val Precision: 0.27857 | Val Recall: 0.58815
  --> Best checkpoint saved (PR-AUC: 0.48532)
Epoch 03/15 | Train Loss: 0.2134 | Val Loss: 0.3956 | Val PR-AUC: 0.50904 | Val ROC-AUC: 0.85878Val Precision: 0.32168 | Val Recall: 0.57605
  --> Best checkpoint saved (PR-AUC: 0.50904)
Epoch 04/15 | Train Loss: 0.1902 | Val Loss: 0.3942 | Val PR-AUC: 0.51622 | Val ROC-AUC: 0.86268Val Precision: 0.27669 | Val Recall: 0.60115
  --> Best checkpoint saved (PR-AUC: 0.51622)
Epoch 05/15 | Train Loss: 0.1723 | Val Loss: 0.4423 | Val PR-AUC: 0.50172 | Val ROC-AUC: 0.84553Val Precision: 0.31219 | Val Recall: 0.56396
E

KeyboardInterrupt: 

In [1]:
import os
import json
import mlflow

# 1. Local path to the downloaded folder
LOCAL_DLRM_DIR = "models/dlrm_folds"

# 2. Point to your existing MLflow experiment
# (Set tracking URI if using DagsHub, or leave default for local ./mlruns)
# mlflow.set_tracking_uri("https://dagshub.com/<user>/<repo>.mlflow")
mlflow.set_experiment("IEEE-fraud-DLRM-time-folds")

with mlflow.start_run(run_name="dlrm_5fold_colab_run") as run:
    # A. Log summary metrics from the saved JSON
    metrics_path = os.path.join(LOCAL_DLRM_DIR, "dlrm_metrics.json")
    if os.path.exists(metrics_path):
        with open(metrics_path, "r") as f:
            summary = json.load(f)
        
        scalar_metrics = {k: v for k, v in summary.items() if isinstance(v, (int, float))}
        mlflow.log_metrics(scalar_metrics)
        print("Logged CV metrics to MLflow.")

    # B. Log training parameters
    mlflow.log_params({
        "architecture": "DLRM",
        "n_splits": 5,
        "embedding_dim": 16,
        "trained_on": "Colab GPU (T4)",
        "eval_metric": "PR-AUC"
    })

    # C. Upload the entire folder of artifacts (.pt models, .joblib preprocessors, .npy arrays)
    mlflow.log_artifacts(LOCAL_DLRM_DIR, artifact_path="dlrm_artifacts")
    print(f"Uploaded all DLRM artifacts to MLflow Run: {run.info.run_id}")

c:\Users\ADMIN\Documents\DataScience\ieee-fraud-detection\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/10/24 10:01:04 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/24 10:01:04 INFO mlflow.store.db.utils: Updating database tables
2026/10/24 10:01:06 INFO mlflow.tracking.fluent: Experiment with name 'IEEE-fraud-DLRM-time-folds' does not exist. Creating a new experiment.


Uploaded all DLRM artifacts to MLflow Run: bf77c563140a4a87b6d0835ae204ee1c
